<a href="https://colab.research.google.com/github/rachana1707-S/Adaptive-MedRAG/blob/feature%2Frachana-retrieval/notebooks/01_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Installing the libraries needed for my retrieval experiments
# huggingface_hub will help me download NFCorpus
# rank-bm25 will be used for the BM25 baseline

!pip -q install huggingface_hub pandas pyarrow numpy rank-bm25

In [6]:
# Importing the libraries that I will use in this notebook

import re
import time

from pathlib import Path

import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download
from rank_bm25 import BM25Okapi

In [7]:
# Creating folders to store the dataset and experiment results

DATA_DIR = Path("data/nfcorpus")
RESULTS_DIR = Path("results")

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [8]:
# Downloading the documents, queries, and test labels
# from Hugging Face instead of the original BEIR server

corpus_path = hf_hub_download(
    repo_id="BeIR/nfcorpus",
    filename="corpus/corpus-00000-of-00001.parquet",
    repo_type="dataset"
)

queries_path = hf_hub_download(
    repo_id="BeIR/nfcorpus",
    filename="queries/queries-00000-of-00001.parquet",
    repo_type="dataset"
)

qrels_path = hf_hub_download(
    repo_id="BeIR/nfcorpus-qrels",
    filename="test.tsv",
    repo_type="dataset"
)

print("All NFCorpus files downloaded successfully!")

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.16MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 80.9kB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

test.tsv:   0%|          | 0.00/280k [00:00<?, ?B/s]

All NFCorpus files downloaded successfully!


In [9]:
# Loading the dataset files into pandas DataFrames

corpus_df = pd.read_parquet(corpus_path)
queries_df = pd.read_parquet(queries_path)
qrels_df = pd.read_csv(qrels_path, sep="\t")

# Checking the number of documents and queries

print("Total documents:", len(corpus_df))
print("Total queries:", len(queries_df))
print("Total relevance labels:", len(qrels_df))

Total documents: 3633
Total queries: 3237
Total relevance labels: 12334


In [10]:
# Looking at the column names before preprocessing the data

print("Corpus columns:", corpus_df.columns.tolist())
print("Query columns:", queries_df.columns.tolist())
print("Qrels columns:", qrels_df.columns.tolist())

# Displaying some documents and queries

display(corpus_df.head(3))
display(queries_df.head(3))
display(qrels_df.head(3))

Corpus columns: ['_id', 'title', 'text']
Query columns: ['_id', 'title', 'text']
Qrels columns: ['query-id', 'corpus-id', 'score']


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."
1,MED-14,Statin use after diagnosis of breast cancer an...,BACKGROUND: Preclinical studies have shown tha...
2,MED-118,Alkylphenols in human milk and their relations...,The aims of this study were to determine the c...


,_id,title,text
0,PLAIN-3,,Breast Cancer Cells Feed on Cholesterol
1,PLAIN-4,,Using Diet to Treat Asthma and Eczema
2,PLAIN-5,,Treating Asthma With Plants vs. Pills


,query-id,corpus-id,score
0,PLAIN-2,MED-2427,2
1,PLAIN-2,MED-10,2
2,PLAIN-2,MED-2429,2


In [11]:
# Using document and query IDs to access the data easily

corpus = {
    str(row["_id"]): {
        "title": str(row["title"] or ""),
        "text": str(row["text"] or "")
    }
    for _, row in corpus_df.fillna("").iterrows()
}

queries = {
    str(row["_id"]): str(row["text"])
    for _, row in queries_df.iterrows()
}

print("Corpus size:", len(corpus))
print("Queries loaded:", len(queries))

Corpus size: 3633
Queries loaded: 3237
